## Here, we'll experiment with such things as:
- How many NULL values are there?
- Are there duplicate rows?
- Are negative prices present?
- Are there any trading days missing?
- Are there timezone problems?
Eventually, these checks will become Python functions inside src/validation

In [11]:
from pathlib import Path
import sys

# Project root = parent directory of the notebooks folder
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

c:\Users\jorda\Documents\Professional\autoTrader\MLPipeline_Jordan


In [ ]:
import pandas as pd
import numpy as np
import os

print("Current working directory:")
print(os.getcwd())

print("\nCurrent directory contents:")
print(os.listdir())

Current working directory:
c:\Users\jorda\Documents\Professional\autoTrader\MLPipeline_Jordan\Notebooks

Current directory contents:
['00_environment_test.ipynb', '01_database_playground.ipynb', '02_validation_playground.ipynb', '03_feature_engineering.ipynb', '04_model_experiments.ipynb', '05_backtesting.ipynb', '06_data_profiling.ipynb', 'practice.ipynb', 'scratch.ipynb']


In [2]:
# For validation:

# validate_required_columns()

# validate_no_missing_dates()

# validate_price_columns()

# validate_volume()

# validate_duplicate_rows()

In [3]:
# Synthetic data generator:
def generate_synthetic_stock_data(
    tickers=("AAA", "BBB", "CCC"),
    trading_days=250,
    start_date="2025-01-01",
    random_seed=42
):
    """
    Generate realistic synthetic OHLCV stock data.
    """
    
    np.random.seed(random_seed)
    
    dates = pd.bdate_range(start=start_date, periods=trading_days)
    
    all_data = []
    
    stock_profiles = {
        "AAA": {"start": 100, "drift": .0005, "volatility": 0.01},
        "BBB": {"start": 50, "drift": .0000, "volatility": 0.018},
        "CCC": {"start": 20, "drift": .0015, "volatility": 0.03}
    }
    
    for ticker in tickers:
        profile = stock_profiles[ticker]
        close = profile["start"]
        
        for date in dates:
            daily_return = np.random.normal(
                profile["drift"],
                profile["volatility"]
            )
            open_price = close
            close = close * (1 + daily_return)
            intraday = abs(np.random.normal(0, profile["volatility"] / 2))
            high = max(open_price, close) * (1 + intraday)
            low = min(open_price, close) * (1 - intraday)
            volume = int(
                np.random.normal(1_000_000, 150_000)
            )
            volume = max(volume, 1000)
            
            all_data.append(
                {
                    "date": date,
                    "ticker": ticker,
                    "open": round(open_price, 2),
                    "high": round(high, 2),
                    "low": round(low, 2),
                    "close": round(close, 2),
                    "volume": volume,
                }
            )
    return pd.DataFrame(all_data)    

In [4]:
# Generate the dataset
df = generate_synthetic_stock_data()

df.head()

,date,ticker,open,high,low,close,volume
0,2025-01-01,AAA,100.00,100.62,99.93,100.55,1097153
1,2025-01-02,AAA,100.55,102.25,100.43,102.13,964879
2,2025-01-03,AAA,102.13,104.19,101.74,103.79,929578
3,2025-01-06,AAA,103.79,104.65,103.55,104.41,930140
4,2025-01-07,AAA,104.41,105.71,103.41,104.71,741262


In [5]:
# Inspect the data
print(df.info())

print()

print(df.describe())

<class 'pandas.DataFrame'>
RangeIndex: 750 entries, 0 to 749
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    750 non-null    datetime64[us]
 1   ticker  750 non-null    str           
 2   open    750 non-null    float64       
 3   high    750 non-null    float64       
 4   low     750 non-null    float64       
 5   close   750 non-null    float64       
 6   volume  750 non-null    int64         
dtypes: datetime64[us](1), float64(4), int64(1), str(1)
memory usage: 41.1 KB
None

                      date        open        high         low       close  \
count                  750  750.000000  750.000000  750.000000  750.000000   
mean   2025-06-24 07:12:00   70.813773   71.743080   70.020547   70.943040   
min    2025-01-01 00:00:00   19.890000   20.400000   19.550000   19.890000   
25%    2025-03-28 00:00:00   37.180000   37.795000   35.870000   37.180000   
50%    2025-06-24 12:00:00   60.36500

In [6]:
# Save the dataset to a CSV file
output_path = Path("C:/Users/jorda/Documents/Professional/autoTrader/MLPipeline_Jordan/data/synthetic/synthetic_stock_data.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

print(f"Saved to: {output_path.resolve()}")

Saved to: C:\Users\jorda\Documents\Professional\autoTrader\MLPipeline_Jordan\data\synthetic\synthetic_stock_data.csv


In [13]:
from src.validation.validate import validate_dataframe

df = pd.read_csv(
    "C:/Users/jorda/Documents/Professional/autoTrader/MLPipeline_Jordan/data/synthetic/synthetic_stock_data.csv",
    parse_dates=["date"]
)

validate_dataframe(df)

print("All validation checks passed!")

All validation checks passed!


In [14]:
import sys

print(sys.path[0])

c:\Users\jorda\Documents\Professional\autoTrader\MLPipeline_Jordan
